# POC House Prices — régression linéaire

Cette analyse utilise uniquement le fichier local `data.csv` (renommé depuis `train.csv`). Le modèle et les métriques sont enregistrés/affichés localement, sans téléchargement de données.

## 1. Charger les bibliothèques

In [ ]:
from pathlib import Path
import pickle

import pandas as pd
import sklearn
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

## 2. Charger `data.csv` dans un dataframe

In [ ]:
project_dir = next(
    (candidate for candidate in (Path.cwd(), Path.cwd().parent) if (candidate / "data.csv").is_file()),
    Path.cwd(),
)
csv_path = project_dir / "data.csv"
if not csv_path.is_file():
    raise FileNotFoundError(
        f"data.csv introuvable dans {Path.cwd()} ni son dossier parent. "
        "Placez data.csv à la racine du projet."
    )

df = pd.read_csv(csv_path)
print(f"Dimensions initiales : {df.shape}")
df.head()

## 3. Garder les variables numériques et supprimer les lignes incomplètes

In [ ]:
numeric_df = df.select_dtypes(include="number").dropna().copy()
if "SalePrice" not in numeric_df.columns:
    raise KeyError("La colonne cible 'SalePrice' est absente des données numériques.")

print(f"Dimensions après sélection numérique et suppression des valeurs manquantes : {numeric_df.shape}")

## 4. Séparer les variables explicatives (X) de la cible (y)

In [ ]:
X = numeric_df.drop(columns="SalePrice")
y = numeric_df["SalePrice"]
print(f"X : {X.shape} ; y : {y.shape}")

## 5. Découper en train et test avec stratification par binning de la cible

In [ ]:
y_bins = pd.qcut(y, q=10, labels=False, duplicates="drop")
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y_bins,
)
print(f"Train : {X_train.shape[0]} lignes ; test : {X_test.shape[0]} lignes")

## 6. Entraîner une régression linéaire

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

## 7. Mesurer RMSE et R² sur train et test

In [ ]:
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

metrics_train_test = pd.DataFrame(
    {
        "Jeu de données": ["Train", "Test"],
        "RMSE": [
            mean_squared_error(y_train, y_train_pred) ** 0.5,
            mean_squared_error(y_test, y_test_pred) ** 0.5,
        ],
        "R²": [r2_score(y_train, y_train_pred), r2_score(y_test, y_test_pred)],
    }
)
display(metrics_train_test)

## 8. Sauvegarder le modèle dans `models/model.pkl`

In [ ]:
model_dir = project_dir / "models"
model_dir.mkdir(exist_ok=True)
model_path = model_dir / "model.pkl"
with model_path.open("wb") as model_file:
    pickle.dump(model, model_file)
print(f"Modèle sauvegardé : {model_path.resolve()}")

## 9–10. Résultats à inscrire au tableau et versions des bibliothèques

Le tableau ci-dessous rassemble les métriques de test demandées (RMSE, MAE, R²) ainsi que les versions de scikit-learn et pandas.

In [ ]:
results_table = pd.DataFrame(
    [
        {
            "RMSE test": mean_squared_error(y_test, y_test_pred) ** 0.5,
            "MAE test": mean_absolute_error(y_test, y_test_pred),
            "R² test": r2_score(y_test, y_test_pred),
            "scikit-learn": sklearn.__version__,
            "pandas": pd.__version__,
        }
    ]
)
display(results_table)